# Polars vs Dask API Demo

Notebook này tập trung vào cú pháp API chính và kết quả mẫu, không đo benchmark phức tạp. Nội dung được tách thành 2 section riêng: Polars trước, Dask sau.


## Setup Chung

Cell này import thư viện, khai báo đường dẫn dữ liệu, và chuẩn bị một file CSV nhỏ để demo API đọc CSV. Đây không phải phần benchmark; mục tiêu là có sẵn `DATA_PATH`, `CSV_PATH`, và thư mục output để các ví dụ Polars/Dask bên dưới chạy được theo cùng một dữ liệu.


In [26]:
from pathlib import Path
import os
import shutil

import polars as pl
import dask
import dask.dataframe as dd

DATA_PATH = Path(os.environ.get(
    "BENCHMARK_DATA_PATH",
    r"D:\Polar vs Dask\data\benchmark_real\1M\part-000.parquet"
))

if not DATA_PATH.exists():
    raise FileNotFoundError(f"Không tìm thấy file dữ liệu: {DATA_PATH}")

OUTPUT_DIR = Path("notebooks/_demo_outputs")
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
CSV_PATH = OUTPUT_DIR / "sample_reviews.csv"
POLARS_OUT_PARQUET = OUTPUT_DIR / "polars_filtered.parquet"
POLARS_OUT_CSV = OUTPUT_DIR / "polars_filtered.csv"
DASK_OUT_PARQUET = OUTPUT_DIR / "dask_filtered.parquet"
DASK_OUT_CSV = OUTPUT_DIR / "dask_filtered.csv"

# Chuẩn bị một CSV nhỏ để demo read_csv / scan_csv.
if not CSV_PATH.exists():
    pl.read_parquet(DATA_PATH).head(1_000).write_csv(CSV_PATH)

print("Polars:", pl.__version__)
print("Dask  :", dask.__version__)
print("Data  :", DATA_PATH)
print("CSV   :", CSV_PATH)


Polars: 0.20.31
Dask  : 2024.5.0
Data  : D:\Polar vs Dask\data\benchmark_real\1M\part-000.parquet
CSV   : notebooks\_demo_outputs\sample_reviews.csv


## Section 1: Polars API


Polars có cả eager `DataFrame` (`read_*`) và lazy `LazyFrame` (`scan_*`). Các ví dụ dưới đây dùng `group_by`, expression API, lazy plan và ghi file.


### 1. Đọc Dữ Liệu

Polars có 2 cách đọc chính. `pl.read_parquet()` và `pl.read_csv()` là eager: dữ liệu được nạp ngay thành `DataFrame`. `pl.scan_parquet()` và `pl.scan_csv()` là lazy: chỉ tạo kế hoạch truy vấn `LazyFrame`, chưa đọc full data cho tới khi gọi `.collect()` hoặc ghi ra file bằng `sink_*`.


In [ ]:
# Polars eager
df = pl.read_parquet(DATA_PATH)

df_csv = pl.read_csv(CSV_PATH)

# Polars lazy, chưa load full data
lf = pl.scan_parquet(DATA_PATH)
lf_csv = pl.scan_csv(CSV_PATH)

print(type(df))
print(type(lf))


<class 'polars.dataframe.frame.DataFrame'>
<class 'polars.lazyframe.frame.LazyFrame'>
shape: (3, 10)
┌───────────┬───────────┬───────────┬───────────┬───┬───────────┬───────────┬───────────┬──────────┐
│ review_id ┆ user_id   ┆ product_i ┆ parent_as ┆ … ┆ review_te ┆ review_ti ┆ helpful_v ┆ verified │
│ ---       ┆ ---       ┆ d         ┆ in        ┆   ┆ xt        ┆ me        ┆ ote       ┆ _purchas │
│ str       ┆ str       ┆ ---       ┆ ---       ┆   ┆ ---       ┆ ---       ┆ ---       ┆ e        │
│           ┆           ┆ str       ┆ str       ┆   ┆ str       ┆ datetime[ ┆ i32       ┆ ---      │
│           ┆           ┆           ┆           ┆   ┆           ┆ ns]       ┆           ┆ bool     │
╞═══════════╪═══════════╪═══════════╪═══════════╪═══╪═══════════╪═══════════╪═══════════╪══════════╡
│ R90942752 ┆ AFKZENTNB ┆ B096S6LZV ┆ B09NSZ5QM ┆ … ┆ Unfortuna ┆ 2023-03-0 ┆ 0         ┆ true     │
│ 77        ┆ Q7A7V7UXW ┆ 4         ┆ F         ┆   ┆ tely      ┆ 4         ┆           ┆  

### 2. Xem Nhanh Dữ Liệu

Các lệnh như `.head()`, `.tail()`, `.shape`, `.columns`, `.dtypes`, `.schema`, `.describe()` và `.null_count()` dùng để kiểm tra nhanh cấu trúc dữ liệu. Với Polars eager `DataFrame`, các kết quả này có sẵn ngay vì dữ liệu đã được nạp vào bộ nhớ.


In [28]:
print("head")
print(df.head(3))

print("tail")
print(df.tail(3))

print("shape:", df.shape)
print("columns:", df.columns)
print("dtypes:", df.dtypes)
print("schema:", df.schema)

print("describe")
print(df.describe())

print("null_count")
print(df.null_count())


head
shape: (3, 10)
┌───────────┬───────────┬───────────┬───────────┬───┬───────────┬───────────┬───────────┬──────────┐
│ review_id ┆ user_id   ┆ product_i ┆ parent_as ┆ … ┆ review_te ┆ review_ti ┆ helpful_v ┆ verified │
│ ---       ┆ ---       ┆ d         ┆ in        ┆   ┆ xt        ┆ me        ┆ ote       ┆ _purchas │
│ str       ┆ str       ┆ ---       ┆ ---       ┆   ┆ ---       ┆ ---       ┆ ---       ┆ e        │
│           ┆           ┆ str       ┆ str       ┆   ┆ str       ┆ datetime[ ┆ i32       ┆ ---      │
│           ┆           ┆           ┆           ┆   ┆           ┆ ns]       ┆           ┆ bool     │
╞═══════════╪═══════════╪═══════════╪═══════════╪═══╪═══════════╪═══════════╪═══════════╪══════════╡
│ R90942752 ┆ AFKZENTNB ┆ B096S6LZV ┆ B09NSZ5QM ┆ … ┆ Unfortuna ┆ 2023-03-0 ┆ 0         ┆ true     │
│ 77        ┆ Q7A7V7UXW ┆ 4         ┆ F         ┆   ┆ tely      ┆ 4         ┆           ┆          │
│           ┆ 5JJI6UGRY ┆           ┆           ┆   ┆ Amazon in ┆ 00:00

### 3. Chọn Cột

Polars dùng `.select()` để chọn cột. Có thể truyền list tên cột hoặc dùng expression như `pl.col("rating")`. Expression API mạnh hơn vì chọn được theo pattern, kiểu dữ liệu, hoặc kết hợp tính toán ngay trong lúc select.


In [29]:
cols = ["product_id", "rating", "helpful_vote"]

print("Chọn bằng list")
print(df.select(cols).head(5))

print("Chọn bằng expression")
print(df.select(
    pl.col("rating"),
    pl.col("helpful_vote")
).head(5))

print("Chọn cột numeric")
print(df.select(pl.col(pl.NUMERIC_DTYPES)).head(5))


Chọn bằng list
shape: (5, 3)
┌────────────┬────────┬──────────────┐
│ product_id ┆ rating ┆ helpful_vote │
│ ---        ┆ ---    ┆ ---          │
│ str        ┆ i8     ┆ i32          │
╞════════════╪════════╪══════════════╡
│ B096S6LZV4 ┆ 3      ┆ 0            │
│ B09KMDBDCN ┆ 3      ┆ 0            │
│ B096N5WK8Q ┆ 4      ┆ 11           │
│ B07JR4QBZ4 ┆ 4      ┆ 26           │
│ B09GY958RK ┆ 2      ┆ 1            │
└────────────┴────────┴──────────────┘
Chọn bằng expression
shape: (5, 2)
┌────────┬──────────────┐
│ rating ┆ helpful_vote │
│ ---    ┆ ---          │
│ i8     ┆ i32          │
╞════════╪══════════════╡
│ 3      ┆ 0            │
│ 3      ┆ 0            │
│ 4      ┆ 11           │
│ 4      ┆ 26           │
│ 2      ┆ 1            │
└────────┴──────────────┘
Chọn cột numeric
shape: (5, 2)
┌────────┬──────────────┐
│ rating ┆ helpful_vote │
│ ---    ┆ ---          │
│ i8     ┆ i32          │
╞════════╪══════════════╡
│ 3      ┆ 0            │
│ 3      ┆ 0            │
│ 4     

### 4. Lọc Dòng

Polars dùng `.filter()` với expression boolean. Nhiều điều kiện kết hợp bằng `&` hoặc `|`, mỗi điều kiện nên đặt trong ngoặc. Lọc null thường dùng `.is_null()` hoặc `.is_not_null()`.


In [30]:
print("rating >= 4")
print(df.filter(pl.col("rating") >= 4).head(5))

print("Nhiều điều kiện")
print(df.filter(
    (pl.col("rating") >= 4) &
    (pl.col("helpful_vote") > 0)
).head(5))

print("Lọc not null")
print(df.filter(pl.col("rating").is_not_null()).head(5))


rating >= 4
shape: (5, 10)
┌───────────┬───────────┬───────────┬───────────┬───┬───────────┬───────────┬───────────┬──────────┐
│ review_id ┆ user_id   ┆ product_i ┆ parent_as ┆ … ┆ review_te ┆ review_ti ┆ helpful_v ┆ verified │
│ ---       ┆ ---       ┆ d         ┆ in        ┆   ┆ xt        ┆ me        ┆ ote       ┆ _purchas │
│ str       ┆ str       ┆ ---       ┆ ---       ┆   ┆ ---       ┆ ---       ┆ ---       ┆ e        │
│           ┆           ┆ str       ┆ str       ┆   ┆ str       ┆ datetime[ ┆ i32       ┆ ---      │
│           ┆           ┆           ┆           ┆   ┆           ┆ ns]       ┆           ┆ bool     │
╞═══════════╪═══════════╪═══════════╪═══════════╪═══╪═══════════╪═══════════╪═══════════╪══════════╡
│ R42577674 ┆ AFKZENTNB ┆ B096N5WK8 ┆ B07RGM3DY ┆ … ┆ I         ┆ 2023-02-0 ┆ 11        ┆ true     │
│ 17        ┆ Q7A7V7UXW ┆ Q         ┆ C         ┆   ┆ purchased ┆ 4         ┆           ┆          │
│           ┆ 5JJI6UGRY ┆           ┆           ┆   ┆ these bc  

### 5. Tạo Cột Mới

Polars dùng `.with_columns()` để thêm hoặc ghi đè cột. Mỗi cột mới thường là một expression kết thúc bằng `.alias("ten_cot")`. Có thể tạo nhiều cột trong cùng một lần gọi để query optimizer xử lý tốt hơn.


In [ ]:
with_score = df.with_columns(
    (pl.col("helpful_vote") / (pl.col("rating") + 1)).alias("helpful_per_rating")
)

print(with_score.select("product_id", "rating", "helpful_vote", "helpful_per_rating").head(5))

with_many = df.with_columns(
    pl.col("rating").cast(pl.Float64).alias("rating_float"),
    (pl.col("rating") >= 4).alias("is_good_rating")
)
print(with_many.select("rating", "rating_float", "is_good_rating").head(5))


shape: (5, 4)
┌────────────┬────────┬──────────────┬────────────────────┐
│ product_id ┆ rating ┆ helpful_vote ┆ helpful_per_rating │
│ ---        ┆ ---    ┆ ---          ┆ ---                │
│ str        ┆ i8     ┆ i32          ┆ f64                │
╞════════════╪════════╪══════════════╪════════════════════╡
│ B096S6LZV4 ┆ 3      ┆ 0            ┆ 0.0                │
│ B09KMDBDCN ┆ 3      ┆ 0            ┆ 0.0                │
│ B096N5WK8Q ┆ 4      ┆ 11           ┆ 2.2                │
│ B07JR4QBZ4 ┆ 4      ┆ 26           ┆ 5.2                │
│ B09GY958RK ┆ 2      ┆ 1            ┆ 0.333333           │
└────────────┴────────┴──────────────┴────────────────────┘
shape: (5, 3)
┌────────┬──────────────┬────────────────┐
│ rating ┆ rating_float ┆ is_good_rating │
│ ---    ┆ ---          ┆ ---            │
│ i8     ┆ f64          ┆ bool           │
╞════════╪══════════════╪════════════════╡
│ 3      ┆ 3.0          ┆ false          │
│ 3      ┆ 3.0          ┆ false          │
│ 4      ┆ 

### 6. Group By / Aggregate

Polars phiên bản mới dùng `.group_by()`, không dùng `.groupby()`. Sau `group_by`, dùng `.agg()` với các expression như mean, sum, count, min, max. `pl.len()` là cách phổ biến để đếm số dòng trong mỗi group.


In [32]:
agg_df = df.group_by("product_id").agg(
    pl.col("rating").mean().alias("avg_rating"),
    pl.col("helpful_vote").sum().alias("total_helpful"),
    pl.len().alias("n_reviews")
)
print(agg_df.sort("n_reviews", descending=True).head(10))

agg_multi = df.group_by(["product_id", "verified_purchase"]).agg(
    pl.col("rating").mean().alias("avg_rating")
)
print(agg_multi.head(10))


shape: (10, 4)
┌────────────┬────────────┬───────────────┬───────────┐
│ product_id ┆ avg_rating ┆ total_helpful ┆ n_reviews │
│ ---        ┆ ---        ┆ ---           ┆ ---       │
│ str        ┆ f64        ┆ i32           ┆ u32       │
╞════════════╪════════════╪═══════════════╪═══════════╡
│ B00S9WXAXA ┆ 4.360806   ┆ 498           ┆ 546       │
│ B06XWJNPT8 ┆ 4.701987   ┆ 25            ┆ 302       │
│ B01KY0E8TO ┆ 4.167832   ┆ 29            ┆ 143       │
│ B07HJB27H8 ┆ 4.360294   ┆ 106           ┆ 136       │
│ B00PHASDRA ┆ 4.75       ┆ 234           ┆ 132       │
│ B01CTMAM04 ┆ 4.25       ┆ 55            ┆ 108       │
│ B007DLVLDY ┆ 4.682243   ┆ 9             ┆ 107       │
│ B07HJ9RX64 ┆ 4.441176   ┆ 33            ┆ 102       │
│ B00459VM6I ┆ 4.755556   ┆ 25            ┆ 90        │
│ B07873S5GR ┆ 4.404494   ┆ 55            ┆ 89        │
└────────────┴────────────┴───────────────┴───────────┘
shape: (10, 3)
┌────────────┬───────────────────┬────────────┐
│ product_id ┆ verified_pu

### 7. Sort Dữ Liệu

Polars dùng `.sort()` cho một hoặc nhiều cột. Tham số `descending=True` áp dụng sort giảm dần; với nhiều cột có thể truyền list boolean như `descending=[False, True]`.


In [ ]:
print(df.sort("rating").select("product_id", "rating").head(5))

print(df.sort("rating", descending=True).select("product_id", "rating").head(5))

print(df.sort(["product_id", "rating"], descending=[False, True])
        .select("product_id", "rating")
        .head(5))

shape: (5, 2)
┌────────────┬────────┐
│ product_id ┆ rating │
│ ---        ┆ ---    │
│ str        ┆ i8     │
╞════════════╪════════╡
│ B003MZ01CM ┆ 1      │
│ B00PUXB1UK ┆ 1      │
│ B00BMKLCH2 ┆ 1      │
│ B07V8MB69G ┆ 1      │
│ B088PNKPKG ┆ 1      │
└────────────┴────────┘
shape: (5, 2)
┌────────────┬────────┐
│ product_id ┆ rating │
│ ---        ┆ ---    │
│ str        ┆ i8     │
╞════════════╪════════╡
│ B07HJ9DJGQ ┆ 5      │
│ B07924VHVP ┆ 5      │
│ B00PHASDRA ┆ 5      │
│ B00D7SVEKC ┆ 5      │
│ B00HF64G7U ┆ 5      │
└────────────┴────────┘
shape: (5, 2)
┌────────────┬────────┐
│ product_id ┆ rating │
│ ---        ┆ ---    │
│ str        ┆ i8     │
╞════════════╪════════╡
│ 0000031895 ┆ 3      │
│ 0000032050 ┆ 5      │
│ 0061208469 ┆ 5      │
│ 0123456479 ┆ 5      │
│ 0123456479 ┆ 5      │
└────────────┴────────┘


### 8. Join Dữ Liệu

Polars dùng `.join()` với `on`, `left_on/right_on`, và `how`. Các kiểu thường gặp gồm `inner`, `left`, `outer`, `semi`, `anti`. Join dùng để gắn thêm thông tin từ bảng khác theo khóa chung như `product_id`.


In [ ]:
product_stats = df.group_by("product_id").agg(
    pl.col("rating").mean().alias("product_avg_rating"),
    pl.len().alias("product_n_reviews")
)

review_sample = df.select("product_id", "rating", "helpful_vote").head(10)
product_sample = product_stats.filter(
    pl.col("product_id").is_in(review_sample["product_id"])
)

df_joined = review_sample.join(
    product_sample,
    on="product_id",
    how="left",
    coalesce=True # gộp cột key join lại, tránh tạo hai cột product_id riêng biệt
)
print(df_joined)

# Các kiểu join phổ biến: how="inner", "left", "outer", "semi", "anti"
# `inner`: chỉ lấy các dòng có key xuất hiện ở cả hai bảng.
# `left`: giữ toàn bộ bảng bên trái, bảng phải có dữ liệu khớp thì ghép vào, không có thì null.
# `outer`: giữ tất cả dòng từ cả hai bảng, bên nào thiếu thì null.
# `semi`: chỉ giữ dòng bên trái nếu key có tồn tại ở bảng phải, nhưng không lấy thêm cột từ bảng phải.
# `anti`: chỉ giữ dòng bên trái nếu key không tồn tại ở bảng phải.


shape: (10, 5)
┌────────────┬────────┬──────────────┬────────────────────┬───────────────────┐
│ product_id ┆ rating ┆ helpful_vote ┆ product_avg_rating ┆ product_n_reviews │
│ ---        ┆ ---    ┆ ---          ┆ ---                ┆ ---               │
│ str        ┆ i8     ┆ i32          ┆ f64                ┆ u32               │
╞════════════╪════════╪══════════════╪════════════════════╪═══════════════════╡
│ B096S6LZV4 ┆ 3      ┆ 0            ┆ 4.333333           ┆ 3                 │
│ B09KMDBDCN ┆ 3      ┆ 0            ┆ 4.0                ┆ 2                 │
│ B096N5WK8Q ┆ 4      ┆ 11           ┆ 4.5                ┆ 2                 │
│ B07JR4QBZ4 ┆ 4      ┆ 26           ┆ 4.540541           ┆ 37                │
│ B09GY958RK ┆ 2      ┆ 1            ┆ 4.4                ┆ 10                │
│ B07HJ9DJGQ ┆ 5      ┆ 0            ┆ 4.266667           ┆ 45                │
│ B07924VHVP ┆ 5      ┆ 0            ┆ 5.0                ┆ 7                 │
│ B00PHASDRA ┆ 5      ┆ 5

### 9. Xử Lý Missing Values

Missing values thường xử lý theo 3 bước: đếm null bằng `.null_count()`, thay null bằng `.fill_null()`, hoặc bỏ dòng null bằng `.drop_nulls()`. Có thể drop null toàn bảng hoặc chỉ theo một vài cột quan trọng.


In [35]:
print("Đếm null")
print(df.null_count())

print("Fill null")
print(df.with_columns(
    pl.col("rating").fill_null(0)
).select("rating").head(5))

print("Drop null toàn bộ")
print(df.drop_nulls().head(3))

print("Drop null theo cột")
print(df.drop_nulls(["rating"]).head(3))


Đếm null
shape: (1, 10)
┌───────────┬─────────┬────────────┬───────────┬───┬───────────┬───────────┬───────────┬───────────┐
│ review_id ┆ user_id ┆ product_id ┆ parent_as ┆ … ┆ review_te ┆ review_ti ┆ helpful_v ┆ verified_ │
│ ---       ┆ ---     ┆ ---        ┆ in        ┆   ┆ xt        ┆ me        ┆ ote       ┆ purchase  │
│ u32       ┆ u32     ┆ u32        ┆ ---       ┆   ┆ ---       ┆ ---       ┆ ---       ┆ ---       │
│           ┆         ┆            ┆ u32       ┆   ┆ u32       ┆ u32       ┆ u32       ┆ u32       │
╞═══════════╪═════════╪════════════╪═══════════╪═══╪═══════════╪═══════════╪═══════════╪═══════════╡
│ 0         ┆ 0       ┆ 0          ┆ 0         ┆ … ┆ 0         ┆ 0         ┆ 0         ┆ 0         │
└───────────┴─────────┴────────────┴───────────┴───┴───────────┴───────────┴───────────┴───────────┘
Fill null
shape: (5, 1)
┌────────┐
│ rating │
│ ---    │
│ i8     │
╞════════╡
│ 3      │
│ 3      │
│ 4      │
│ 4      │
│ 2      │
└────────┘
Drop null toàn bộ
shape

### 10. Ép Kiểu Dữ Liệu

Polars ép kiểu bằng expression `.cast()`, thường đặt trong `.with_columns()`. Với ngày tháng dạng string, dùng `.str.strptime()`; nếu cột đã là Datetime thì có thể cast trực tiếp sang `pl.Date`.


In [36]:
casted = df.with_columns(
    pl.col("rating").cast(pl.Float64).alias("rating_float"),
    pl.col("helpful_vote").cast(pl.Int64).alias("helpful_vote_i64"),
)
print(casted.select("rating_float", "helpful_vote_i64").head(5))

# Nếu ngày đang là string, dùng str.strptime. Ở dataset này review_time đã là Datetime.
dates = df.select(pl.col("review_time").cast(pl.Date).alias("review_date"))
print(dates.head(5))


shape: (5, 2)
┌──────────────┬──────────────────┐
│ rating_float ┆ helpful_vote_i64 │
│ ---          ┆ ---              │
│ f64          ┆ i64              │
╞══════════════╪══════════════════╡
│ 3.0          ┆ 0                │
│ 3.0          ┆ 0                │
│ 4.0          ┆ 11               │
│ 4.0          ┆ 26               │
│ 2.0          ┆ 1                │
└──────────────┴──────────────────┘
shape: (5, 1)
┌─────────────┐
│ review_date │
│ ---         │
│ date        │
╞═════════════╡
│ 2023-03-04  │
│ 2023-02-22  │
│ 2023-02-04  │
│ 2018-12-18  │
│ 2022-02-18  │
└─────────────┘


### 11. Làm Việc Với String

Polars xử lý chuỗi qua namespace `.str`, ví dụ `.str.to_lowercase()`, `.str.contains()`, `.str.len_chars()`. Các thao tác này có thể dùng trong `.with_columns()`, `.filter()`, hoặc `.select()`.


In [37]:
text_df = df.with_columns(
    pl.col("review_text").str.to_lowercase().alias("review_text_lower")
)
print(text_df.select("review_text_lower").head(3))

contains_good = df.filter(
    pl.col("review_text").str.contains("good", literal=True)
)
print(contains_good.select("review_text").head(3))

text_length = df.with_columns(
    pl.col("review_text").str.len_chars().alias("text_length")
)
print(text_length.select("review_text", "text_length").head(3))


shape: (3, 1)
┌─────────────────────────────────┐
│ review_text_lower               │
│ ---                             │
│ str                             │
╞═════════════════════════════════╡
│ unfortunately amazon in their … │
│ useless under 40 degrees unles… │
│ i purchased these bc they are … │
└─────────────────────────────────┘
shape: (3, 1)
┌─────────────────────────────────┐
│ review_text                     │
│ ---                             │
│ str                             │
╞═════════════════════════════════╡
│ I purchased these bc they are … │
│ You guys need to get this hat!… │
│ If you only care about the pho… │
└─────────────────────────────────┘
shape: (3, 2)
┌─────────────────────────────────┬─────────────┐
│ review_text                     ┆ text_length │
│ ---                             ┆ ---         │
│ str                             ┆ u32         │
╞═════════════════════════════════╪═════════════╡
│ Unfortunately Amazon in their … ┆ 1719        │
│ Useless 

### 12. Lazy Execution

Lazy execution là điểm mạnh của Polars. Ta build query bằng `pl.scan_*()`, nối filter/select/group_by/agg/sort, sau đó xem kế hoạch bằng `.explain(optimized=True)`. Query chỉ chạy thật khi gọi `.collect()` hoặc `sink_*`.


In [38]:
query = (
    pl.scan_parquet(DATA_PATH)
    .filter(pl.col("rating").is_not_null())
    .select(["product_id", "rating", "helpful_vote"])
    .group_by("product_id")
    .agg(
        pl.col("rating").mean().alias("avg_rating"),
        pl.col("helpful_vote").sum().alias("total_helpful"),
    )
    .sort("total_helpful", descending=True)
)

print("Optimized execution plan")
print(query.explain(optimized=True))

result = query.collect()
print(result.head(10))


Optimized execution plan
SORT BY [col("total_helpful")]
  AGGREGATE
  	[col("rating").mean().alias("avg_rating"), col("helpful_vote").sum().alias("total_helpful")] BY [col("product_id")] FROM
    simple π 3/3 ["product_id", "rating", ... 1 other column]

        Parquet SCAN D:\Polar vs Dask\data\benchmark_real\1M\part-000.parquet
        PROJECT 3/10 COLUMNS
        SELECTION: col("rating").is_not_null()
shape: (10, 3)
┌────────────┬────────────┬───────────────┐
│ product_id ┆ avg_rating ┆ total_helpful │
│ ---        ┆ ---        ┆ ---           │
│ str        ┆ f64        ┆ i32           │
╞════════════╪════════════╪═══════════════╡
│ B07PLW8LPK ┆ 4.6        ┆ 3271          │
│ B01LZ75ROQ ┆ 4.8        ┆ 1953          │
│ B00T6TTEU8 ┆ 4.285714   ┆ 1750          │
│ B00BLXOEDY ┆ 5.0        ┆ 1629          │
│ B01GNMMQBY ┆ 4.4        ┆ 1424          │
│ B08KRXXZKT ┆ 5.0        ┆ 1216          │
│ B077MHG3YW ┆ 4.0        ┆ 1208          │
│ B00KUY11EK ┆ 4.0        ┆ 970           │
│ B0

## Section 2: Dask API


Dask DataFrame lazy theo mặc định. Kết quả chỉ được tính khi gọi `.compute()`, `.head()`, hoặc khi ghi file.


### 1. Đọc Dữ Liệu

Dask đọc dữ liệu theo kiểu lazy mặc định. `dd.read_parquet()` và `dd.read_csv()` tạo một Dask DataFrame gồm nhiều partition và chưa tính toàn bộ dữ liệu. Các thao tác chỉ thực thi khi gọi `.compute()`, `.head()`, `.tail()`, hoặc ghi ra file.


In [41]:
ddf = dd.read_parquet(DATA_PATH, engine="pyarrow")
ddf_csv = dd.read_csv(CSV_PATH)

print(type(ddf))
print("npartitions:", ddf.npartitions)
print(ddf.head(3))
print(ddf_csv.head(3))


<class 'dask_expr._collection.DataFrame'>
npartitions: 1
     review_id                       user_id  product_id parent_asin  rating  \
0  R9094275277  AFKZENTNBQ7A7V7UXW5JJI6UGRYQ  B096S6LZV4  B09NSZ5QMF       3   
1  R4069340459  AFKZENTNBQ7A7V7UXW5JJI6UGRYQ  B09KMDBDCN  B08NGL3X17       3   
2  R4257767417  AFKZENTNBQ7A7V7UXW5JJI6UGRYQ  B096N5WK8Q  B07RGM3DYC       4   

                              review_title  \
0  Arrived Damaged : liquid in hub locker!   
1                Useless under 40 degrees.   
2   Not waterproof, but a very comfy shoe.   

                                         review_text review_time  \
0  Unfortunately Amazon in their wisdom (cough, c...  2023-03-04   
1  Useless under 40 degrees unless you’re just ru...  2023-02-22   
2  I purchased these bc they are supposed to be w...  2023-02-04   

   helpful_vote  verified_purchase  
0             0               True  
1             0              False  
2            11               True  
     review_id  

### 2. Xem Nhanh Dữ Liệu

Dask giữ nhiều thông tin metadata như columns và dtypes mà chưa cần tính full data. Tuy nhiên số dòng, `describe()`, null count, hoặc các thống kê thật cần `.compute()` vì Dask phải chạy graph trên dữ liệu.


In [42]:
print("head")
print(ddf.head(3))

print("tail")
print(ddf.tail(3))

print("shape lazy:", ddf.shape)
print("row count computed:", ddf.shape[0].compute())
print("columns:", list(ddf.columns))
print("dtypes:")
print(ddf.dtypes)

print("describe")
print(ddf.describe().compute())

print("null_count")
print(ddf.isnull().sum().compute())


head
     review_id                       user_id  product_id parent_asin  rating  \
0  R9094275277  AFKZENTNBQ7A7V7UXW5JJI6UGRYQ  B096S6LZV4  B09NSZ5QMF       3   
1  R4069340459  AFKZENTNBQ7A7V7UXW5JJI6UGRYQ  B09KMDBDCN  B08NGL3X17       3   
2  R4257767417  AFKZENTNBQ7A7V7UXW5JJI6UGRYQ  B096N5WK8Q  B07RGM3DYC       4   

                              review_title  \
0  Arrived Damaged : liquid in hub locker!   
1                Useless under 40 degrees.   
2   Not waterproof, but a very comfy shoe.   

                                         review_text review_time  \
0  Unfortunately Amazon in their wisdom (cough, c...  2023-03-04   
1  Useless under 40 degrees unless you’re just ru...  2023-02-22   
2  I purchased these bc they are supposed to be w...  2023-02-04   

   helpful_vote  verified_purchase  
0             0               True  
1             0              False  
2            11               True  
tail
          review_id                       user_id  product_id p

### 3. Chọn Cột

Dask chọn cột giống pandas: dùng `ddf[[...]]` hoặc lọc danh sách cột. Với chọn theo kiểu dữ liệu, `select_dtypes()` giúp lấy các cột numeric, string, datetime trước khi compute.


In [43]:
cols = ["product_id", "rating", "helpful_vote"]

print("Chọn bằng list")
print(ddf[cols].head(5))

print("Chọn numeric columns")
numeric_cols = ddf.select_dtypes(include="number").columns.tolist()
print(numeric_cols)
print(ddf[numeric_cols].head(5))


Chọn bằng list
   product_id  rating  helpful_vote
0  B096S6LZV4       3             0
1  B09KMDBDCN       3             0
2  B096N5WK8Q       4            11
3  B07JR4QBZ4       4            26
4  B09GY958RK       2             1
Chọn numeric columns
['rating', 'helpful_vote']
   rating  helpful_vote
0       3             0
1       3             0
2       4            11
3       4            26
4       2             1


### 4. Lọc Dòng

Dask lọc dòng bằng boolean indexing giống pandas. Vì Dask lazy, biểu thức lọc chỉ thêm bước vào task graph; dữ liệu thật chỉ được lọc khi gọi `.compute()` hoặc `.head()`.


In [44]:
print("rating >= 4")
print(ddf[ddf["rating"] >= 4].head(5))

print("Nhiều điều kiện")
print(ddf[
    (ddf["rating"] >= 4) &
    (ddf["helpful_vote"] > 0)
].head(5))

print("Lọc not null")
print(ddf[ddf["rating"].notnull()].head(5))


rating >= 4
     review_id                       user_id  product_id parent_asin  rating  \
2  R4257767417  AFKZENTNBQ7A7V7UXW5JJI6UGRYQ  B096N5WK8Q  B07RGM3DYC       4   
3  R2377921174  AFKZENTNBQ7A7V7UXW5JJI6UGRYQ  B07JR4QBZ4  B07BWS4CSM       4   
5  R7422144694  AGGZ357AO26RQZVRLGU4D4N52DZQ  B07HJ9DJGQ  B07HJ84J9M       5   
6  R5452841325  AGGZ357AO26RQZVRLGU4D4N52DZQ  B07924VHVP  B01DDC83C8       5   
7  R4358597735  AGGZ357AO26RQZVRLGU4D4N52DZQ  B00PHASDRA  B0BNP511CS       5   

                             review_title  \
2  Not waterproof, but a very comfy shoe.   
3       Lovely, but QA issues with sewing   
5                  Great watch for nurses   
6                 Perfect Ren Fest dress!   
7                FINALLY, my hat soulmate   

                                         review_text review_time  \
2  I purchased these bc they are supposed to be w...  2023-02-04   
3  I’ll start by saying I love this robe!  I trul...  2018-12-18   
5  Love this watch! It's perfect

### 5. Tạo Cột Mới

Dask thường dùng `.assign()` để tạo cột mới theo phong cách pandas. Các phép tính trên cột vẫn lazy và được đưa vào graph; kết quả chỉ xuất hiện đầy đủ khi compute.


In [45]:
ddf_with_score = ddf.assign(
    helpful_per_rating=ddf["helpful_vote"] / (ddf["rating"] + 1)
)
print(ddf_with_score[["product_id", "rating", "helpful_vote", "helpful_per_rating"]].head(5))

ddf_with_many = ddf.assign(
    rating_float=ddf["rating"].astype("float64"),
    is_good_rating=ddf["rating"] >= 4,
)
print(ddf_with_many[["rating", "rating_float", "is_good_rating"]].head(5))


   product_id  rating  helpful_vote  helpful_per_rating
0  B096S6LZV4       3             0            0.000000
1  B09KMDBDCN       3             0            0.000000
2  B096N5WK8Q       4            11            2.200000
3  B07JR4QBZ4       4            26            5.200000
4  B09GY958RK       2             1            0.333333
   rating  rating_float  is_good_rating
0       3           3.0           False
1       3           3.0           False
2       4           4.0            True
3       4           4.0            True
4       2           2.0           False


### 6. Group By / Aggregate

Dask dùng `.groupby()` giống pandas. Aggregation tạo graph phân tán theo partition; với dữ liệu lớn, bước groupby có thể cần shuffle hoặc combine nhiều partition trước khi `.compute()` trả kết quả cuối.


In [46]:
dask_agg = ddf.groupby("product_id").agg({
    "rating": "mean",
    "helpful_vote": "sum",
})
# Dask đặt tên cột theo input, nên đổi tên sau compute cho dễ đọc.
dask_agg = dask_agg.rename(columns={
    "rating": "avg_rating",
    "helpful_vote": "total_helpful",
})
counts = ddf.groupby("product_id").size().rename("n_reviews")
print(dask_agg.join(counts).compute().sort_values("n_reviews", ascending=False).head(10))

dask_multi = ddf.groupby(["product_id", "verified_purchase"])["rating"].mean().rename("avg_rating")
print(dask_multi.compute().head(10))


            avg_rating  total_helpful  n_reviews
product_id                                      
B00S9WXAXA    4.360806            498        546
B06XWJNPT8    4.701987             25        302
B01KY0E8TO    4.167832             29        143
B07HJB27H8    4.360294            106        136
B00PHASDRA    4.750000            234        132
B01CTMAM04    4.250000             55        108
B007DLVLDY    4.682243              9        107
B07HJ9RX64    4.441176             33        102
B00459VM6I    4.755556             25         90
B07873S5GR    4.404494             55         89
product_id  verified_purchase
0000031895  True                 3.000000
0000032050  True                 5.000000
0061208469  True                 5.000000
0123456479  True                 5.000000
0152053077  False                4.000000
0307743691  False                4.000000
            True                 4.000000
0307931528  False                5.000000
            True                 4.863636
0310

### 7. Sort Dữ Liệu

Dask dùng `.sort_values()` giống pandas. Sort toàn cục trên dữ liệu lớn có thể tốn chi phí vì phải sắp xếp qua nhiều partition; khi demo nên dùng `.head()` để chỉ hiển thị vài dòng kết quả.


In [47]:
print(ddf.sort_values("rating")[["product_id", "rating"]].head(5))

print(ddf.sort_values("rating", ascending=False)[["product_id", "rating"]].head(5))

print(ddf.sort_values(["product_id", "rating"], ascending=[True, False])
        [["product_id", "rating"]]
        .head(5))


        product_id  rating
463289  B07VGDFNMF       1
717374  B01M6886NC       1
567675  B07X17ZND1       1
567677  B07MXTJQDS       1
302772  B01E2YQDHI       1
        product_id  rating
379757  B07BY394KT       5
449052  B07QGVMS2Y       5
449083  B071H5KQLH       5
449085  B076TQ457H       5
449087  B01BTL7FVU       5
        product_id  rating
402074  0000031895       3
70829   0000032050       5
483417  0061208469       5
121362  0123456479       5
275441  0123456479       5


### 8. Join Dữ Liệu

Dask dùng `.merge()` để join theo phong cách pandas. Các kiểu phổ biến gồm `inner`, `left`, `right`, `outer`. Semi/anti join thường được mô phỏng bằng `isin()` và filter thay vì là mode trực tiếp.


In [48]:
dask_product_stats = ddf.groupby("product_id")["rating"].mean().rename("product_avg_rating").reset_index()
dask_review_sample = ddf[["product_id", "rating", "helpful_vote"]].head(10)
dask_sample_ids = dask_review_sample["product_id"].tolist()
dask_product_sample = dask_product_stats[dask_product_stats["product_id"].isin(dask_sample_ids)].compute()

dask_joined = dd.from_pandas(dask_review_sample, npartitions=1).merge(
    dd.from_pandas(dask_product_sample, npartitions=1),
    on="product_id",
    how="left"
)
print(dask_joined.compute())

# Các kiểu join phổ biến: how="inner", "left", "outer".
# Semi/anti join trong Dask thường làm bằng isin/filter.


   product_id  rating  helpful_vote  product_avg_rating
0  B096S6LZV4       3             0            4.333333
1  B09KMDBDCN       3             0            4.000000
2  B096N5WK8Q       4            11            4.500000
3  B07JR4QBZ4       4            26            4.540541
4  B09GY958RK       2             1            4.400000
5  B07HJ9DJGQ       5             0            4.266667
6  B07924VHVP       5             0            5.000000
7  B00PHASDRA       5            52            4.750000
8  B01KY0E8TO       2             3            4.167832
9  B00S9WXAXA       3             5            4.360806


### 9. Xử Lý Missing Values

Dask dùng API gần pandas: `.isnull().sum()`, `.fillna()`, `.dropna()`. Vì các phép đếm null cần quét dữ liệu, thường phải gọi `.compute()` để lấy kết quả thật.


In [49]:
print("Đếm null")
print(ddf.isnull().sum().compute())

print("Fill null")
print(ddf.assign(rating=ddf["rating"].fillna(0))[["rating"]].head(5))

print("Drop null toàn bộ")
print(ddf.dropna().head(3))

print("Drop null theo cột")
print(ddf.dropna(subset=["rating"]).head(3))


Đếm null
review_id            0
user_id              0
product_id           0
parent_asin          0
rating               0
review_title         0
review_text          0
review_time          0
helpful_vote         0
verified_purchase    0
dtype: int64
Fill null
   rating
0       3
1       3
2       4
3       4
4       2
Drop null toàn bộ
     review_id                       user_id  product_id parent_asin  rating  \
0  R9094275277  AFKZENTNBQ7A7V7UXW5JJI6UGRYQ  B096S6LZV4  B09NSZ5QMF       3   
1  R4069340459  AFKZENTNBQ7A7V7UXW5JJI6UGRYQ  B09KMDBDCN  B08NGL3X17       3   
2  R4257767417  AFKZENTNBQ7A7V7UXW5JJI6UGRYQ  B096N5WK8Q  B07RGM3DYC       4   

                              review_title  \
0  Arrived Damaged : liquid in hub locker!   
1                Useless under 40 degrees.   
2   Not waterproof, but a very comfy shoe.   

                                         review_text review_time  \
0  Unfortunately Amazon in their wisdom (cough, c...  2023-03-04   
1  Useless under 4

### 10. Ép Kiểu Dữ Liệu

Dask ép kiểu bằng `.astype()` giống pandas. Với ngày tháng string, thường dùng `dd.to_datetime()`; với cột datetime sẵn có, có thể dùng accessor `.dt` để lấy date, year, month.


In [50]:
dask_casted = ddf.assign(
    rating_float=ddf["rating"].astype("float64"),
    helpful_vote_i64=ddf["helpful_vote"].astype("int64"),
)
print(dask_casted[["rating_float", "helpful_vote_i64"]].head(5))

# Nếu ngày đang là string, dùng dd.to_datetime. Ở dataset này review_time đã là datetime64.
dask_dates = ddf.assign(review_date=ddf["review_time"].dt.date)
print(dask_dates[["review_date"]].head(5))


   rating_float  helpful_vote_i64
0           3.0                 0
1           3.0                 0
2           4.0                11
3           4.0                26
4           2.0                 1
  review_date
0  2023-03-04
1  2023-02-22
2  2023-02-04
3  2018-12-18
4  2022-02-18


### 11. Làm Việc Với String

Dask hỗ trợ nhiều thao tác string qua `.str` giống pandas, ví dụ `.str.lower()`, `.str.contains()`, `.str.len()`. Các thao tác này vẫn lazy và chạy theo partition khi compute.


In [51]:
dask_text = ddf.assign(
    review_text_lower=ddf["review_text"].str.lower()
)
print(dask_text[["review_text_lower"]].head(3))

contains_good = ddf[ddf["review_text"].str.contains("good", regex=False, na=False)]
print(contains_good[["review_text"]].head(3))

dask_text_length = ddf.assign(
    text_length=ddf["review_text"].str.len()
)
print(dask_text_length[["review_text", "text_length"]].head(3))


                                   review_text_lower
0  unfortunately amazon in their wisdom (cough, c...
1  useless under 40 degrees unless you’re just ru...
2  i purchased these bc they are supposed to be w...
                                          review_text
2   I purchased these bc they are supposed to be w...
10  You guys need to get this hat! I'm a petite ad...
39  If you only care about the photos looking good...
                                         review_text  text_length
0  Unfortunately Amazon in their wisdom (cough, c...         1719
1  Useless under 40 degrees unless you’re just ru...         2305
2  I purchased these bc they are supposed to be w...         1951


### 12. Lazy Execution

Dask DataFrame luôn xây task graph cho các thao tác. Có thể xem số task bằng `.__dask_graph__()`. Graph chỉ chạy khi gọi `.compute()`, `.persist()`, `.head()`, hoặc ghi dữ liệu ra file.


In [52]:
dask_query = (
    ddf[ddf["rating"].notnull()][["product_id", "rating", "helpful_vote"]]
    .groupby("product_id")
    .agg({"rating": "mean", "helpful_vote": "sum"})
    .rename(columns={"rating": "avg_rating", "helpful_vote": "total_helpful"})
)

print("Dask graph tasks:", len(dask_query.__dask_graph__()))

result_dask = dask_query.compute().sort_values("total_helpful", ascending=False)
print(result_dask.head(10))


Dask graph tasks: 8
            avg_rating  total_helpful
product_id                           
B07PLW8LPK    4.600000           3271
B01LZ75ROQ    4.800000           1953
B00T6TTEU8    4.285714           1750
B00BLXOEDY    5.000000           1629
B01GNMMQBY    4.400000           1424
B08KRXXZKT    5.000000           1216
B077MHG3YW    4.000000           1208
B00KUY11EK    4.000000            970
B01M06EL47    5.000000            947
B07F6F12V2    4.500000            924
